In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoConfig, PreTrainedModel, Trainer, TrainingArguments
import torch, torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
from src.preprocess import QuestPreprocessor
from src.postprocess import QuestPostprocessor
from src.loss import SpearmanLoss

MODEL_NAME = "answerdotai/ModernBERT-base"
TARGET_COLS = ['question_asker_intent_understanding',
 'question_body_critical',
 'question_conversational',
 'question_expect_short_answer',
 'question_fact_seeking',
 'question_has_commonly_accepted_answer',
 'question_interestingness_others',
 'question_interestingness_self',
 'question_multi_intent',
 'question_not_really_a_question',
 'question_opinion_seeking',
 'question_type_choice',
 'question_type_compare',
 'question_type_consequence',
 'question_type_definition',
 'question_type_entity',
 'question_type_instructions',
 'question_type_procedure',
 'question_type_reason_explanation',
 'question_type_spelling',
 'question_well_written',
 'answer_helpful',
 'answer_level_of_information',
 'answer_plausible',
 'answer_relevance',
 'answer_satisfaction',
 'answer_type_instructions',
 'answer_type_procedure',
 'answer_type_reason_explanation',
 'answer_well_written']

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class QuestDataset(torch.utils.data.Dataset):
	def __init__(self, df, tokenizer, max_len=512):
		self.df = df
		self.tokenizer = tokenizer
		self.max_len = max_len
		# We assume 'text_input' column is already created by the preprocessor

	def __len__(self):
		return len(self.df)

	def __getitem__(self, idx):
		row = self.df.iloc[idx]

		# 1) Get the pre-processed text
		# Ensure it's a string in case of NaNs
		text = str(row['text_input'])

		# 2) Tokenize
		enc = self.tokenizer(
			text,
			truncation=True,
			padding="max_length",
			max_length=self.max_len,
			return_tensors="pt"
		)
		item = {k: v.squeeze() for k, v in enc.items()}

		# 3) Labels (Only if they exist in the dataframe)
		# We check if the first target column exists
		if TARGET_COLS[0] in row:
			labels = torch.tensor(row[TARGET_COLS].astype(float).values, dtype=torch.float32)
			item["labels"] = labels

		return item

class CustomModel(PreTrainedModel):
	def __init__(self, config, n_targets=30, dropout_prob=0.1, freeze_encoder=False):
		super().__init__(config)
		if config._name_or_path:
			self.encoder = AutoModel.from_pretrained(config._name_or_path)
		else:
			self.encoder = AutoModel.from_config(config)
		# freeze encoder if needed
		if freeze_encoder:
			for param in self.encoder.parameters():
				param.requires_grad = False

		self.dropout = nn.Dropout(dropout_prob)
		self.regressor = nn.Linear(config.hidden_size, n_targets)
		self.sigmoid = nn.Sigmoid()
		self.spearman_loss = SpearmanLoss()
		# init weights
		self.post_init()

	def forward(self, input_ids=None, attention_mask=None, labels=None, token_type_ids=None):
		outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
		# some models have pooler_output, some don't
		if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
			pooled_output = outputs.pooler_output
		else:
			pooled_output = outputs.last_hidden_state[:, 0, :]

		x = self.dropout(pooled_output)

		raw_logits = self.regressor(x)

		preds = self.sigmoid(raw_logits)
		loss = None
		if labels is not None:
			#loss_mse = nn.MSELoss()(preds, labels)
			loss_bce = nn.BCEWithLogitsLoss()(raw_logits, labels)
			loss_spearman = self.spearman_loss(preds, labels)
			#loss = loss_mse
			loss = loss_bce * 0.9 + loss_spearman * 0.1

		return {"loss": loss, "logits": preds}

config = AutoConfig.from_pretrained(MODEL_NAME)
model = CustomModel(config, n_targets=len(TARGET_COLS))

In [5]:
import torch
from torch.utils.data import DataLoader, default_collate

# 1) Make sure your test‐only Dataset returns *only* tensors:
class QuestTestDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__():
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Use the preprocessed text_input column
        text = str(row['text_input'])
        
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        # squeeze batch‐dim, and drop anything that isn’t a tensor
        return {k: v.squeeze() for k, v in enc.items()}


In [ ]:
# --- ADDED TRAINING BLOCK ---
import numpy as np
from scipy.stats import spearmanr

# 1. Load Train Data
print('Loading train.csv...')
raw_train_df = pd.read_csv('train.csv')

# --- NEW PREPROCESSING STEP ---
preprocessor = QuestPreprocessor(use_category=True, use_host=True)
train_df_processed = preprocessor.preprocess(raw_train_df)
# ------------------------------

postprocessor = QuestPostprocessor(train_labels=raw_train_df[TARGET_COLS].values)


train_split, val_split = train_test_split(train_df_processed, test_size=0.1, random_state=42)

train_ds = QuestDataset(train_split, tokenizer)
val_ds = QuestDataset(val_split, tokenizer)

# 2. Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    logits = postprocessor.optimize_ranks(logits)
    
    rhos = []
    for i in range(labels.shape[1]):
        if np.std(labels[:, i]) == 0 or np.std(logits[:, i]) == 0:
            rhos.append(0.0)
        else:
            rho, _ = spearmanr(labels[:, i], logits[:, i])
            rhos.append(rho)
    return {'spearman': np.nanmean(rhos)}

# 3. Train
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=12,
    per_device_eval_batch_size=12,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=50,
    #eval_strategy='epoch',
    eval_strategy='steps',
    eval_steps=80,
    save_strategy='steps',
    save_steps=80,
    save_total_limit=5,
    load_best_model_at_end=True,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print('Starting Training...')
trainer.train()
print('Training Complete. Evaluating...')
eval_res = trainer.evaluate()
print(f'Validation Spearman Rho: {eval_res["eval_spearman"]:.4f}')


Loading train.csv...
Starting Training...


Step,Training Loss,Validation Loss,Spearman
80,0.757500,0.586460,0.248073
160,0.569300,0.548464,0.308530
240,0.545500,0.529743,0.335810
320,0.533600,0.513371,0.360004


In [ ]:

# 2) Load test.csv and wrap it
test_df = pd.read_csv("test.csv")
test_df_processed = preprocessor.preprocess(test_df)
test_ds = QuestDataset(test_df_processed, tokenizer)


In [ ]:

# 3) Build a DataLoader
loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    collate_fn=default_collate
)


In [ ]:

# 4) Move model to device and eval
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 5) Run through batches and collect
all_logits = []
from tqdm import tqdm
with torch.no_grad():
    for batch in tqdm(loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        batch.pop("token_type_ids", None)
        out = model(**batch)
        all_logits.append(out["logits"].cpu())

# 6) Concatenate & sigmoid already applied in your head
preds = torch.cat(all_logits, dim=0).numpy()


  0%|          | 0/60 [00:00<?, ?it/s]

100%|██████████| 60/60 [00:20<00:00,  2.88it/s]


In [ ]:
# 7) Build submission
preds = postprocessor.optimize_ranks(preds)
sub = pd.DataFrame(preds, columns=TARGET_COLS)
sub.insert(0, "qa_id", test_df["qa_id"])
sub.to_csv("submission.csv", index=False)